In [ ]:
"""
Task 9: Real-Time Vector Stream Ingestion & Dynamic Reindexing
Simplified, runnable simulation. A real deployment replaces:
  - `FakeKafkaQueue`      -> Apache Kafka (kafka-python / confluent-kafka)
  - `process_batch()`     -> a PySpark Structured Streaming `foreachBatch`
  - `VectorStore.upsert`  -> Qdrant's `client.upsert(...)` REST/gRPC call
The batching / vectorize / upsert control flow is identical either way.
"""

import queue
import threading
import time
import numpy as np
from sentence_transformers import SentenceTransformer  # or any embedding model


class FakeKafkaQueue:
    """Stand-in producer/consumer queue for a Kafka topic."""
    def __init__(self):
        self.q = queue.Queue()

    def produce(self, message):
        self.q.put(message)

    def poll_batch(self, batch_size=10, timeout=1.0):
        batch = []
        deadline = time.time() + timeout
        while len(batch) < batch_size and time.time() < deadline:
            try:
                batch.append(self.q.get(timeout=0.1))
            except queue.Empty:
                break
        return batch


class VectorStore:
    """Stand-in for a Qdrant collection with zero-downtime upserts."""
    def __init__(self):
        self.vectors = {}
        self.lock = threading.Lock()

    def upsert(self, ids, embeddings, payloads):
        with self.lock:                      # readers never see a half-written batch
            for i, emb, payload in zip(ids, embeddings, payloads):
                self.vectors[i] = {"embedding": emb, "payload": payload}

    def search(self, query_vec, k=3):
        with self.lock:
            items = list(self.vectors.items())
        sims = [(i, np.dot(query_vec, v["embedding"]) /
                 (np.linalg.norm(query_vec) * np.linalg.norm(v["embedding"]) + 1e-9))
                for i, v in items]
        sims.sort(key=lambda x: -x[1])
        return sims[:k]


def spark_style_process_batch(batch, model, store, batch_id):
    """Equivalent of a PySpark `foreachBatch(process_batch)` micro-batch handler."""
    if not batch:
        return
    texts = [msg["text"] for msg in batch]
    ids = [msg["id"] for msg in batch]
    embeddings = model.encode(texts, normalize_embeddings=True)
    store.upsert(ids, embeddings, [{"text": t} for t in texts])
    print(f"[batch {batch_id}] ingested {len(batch)} records, "
          f"store size = {len(store.vectors)}")


def run_streaming_pipeline(duration_sec=2):
    kafka = FakeKafkaQueue()
    store = VectorStore()
    model = SentenceTransformer("all-MiniLM-L6-v2")

    def producer():
        for i in range(50):
            kafka.produce({"id": i, "text": f"log entry number {i} about system status"})
            time.sleep(0.02)

    threading.Thread(target=producer, daemon=True).start()

    batch_id = 0
    start = time.time()
    while time.time() - start < duration_sec:
        batch = kafka.poll_batch(batch_size=10, timeout=0.5)
        spark_style_process_batch(batch, model, store, batch_id)
        batch_id += 1

    return store


if __name__ == "__main__":
    store = run_streaming_pipeline()
    query = np.random.randn(384)  # MiniLM embedding dim
    print("\nSample search:", store.search(query))

---
## Task 9: Real-Time Vector Stream Ingestion & Dynamic Reindexing